# Modelado de Fatiga con PatchTST (Patch Time Series Transformer)

Este notebook contiene la explicación teórica, la revisión de literatura científica, la arquitectura detallada y la implementación paso a paso del modelo **PatchTST** en PyTorch (empleando segmentación por parches y canales independientes de forma manual) para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

La arquitectura **PatchTST** (Nie et al., 2022) representa un hito en la adaptación de los Transformers para series temporales. A diferencia del Transformer clásico, que procesa timesteps individuales y acopla todos los canales en un vector conjunto, PatchTST introduce dos principios fundamentales:

### A. Independencia de Canales (Channel Independence)

El modelo trata cada variable fisiológica (canales) como una serie temporal univariada totalmente independiente. Dado un lote de entrada $X \in \mathbb{R}^{B \times L \times M}$:
1. Permutamos a la forma de canales: $X_{trans} \in \mathbb{R}^{B \times M \times L}$.
2. Aplanamos las dimensiones de lote y canal para formar un lote ampliado de series univariadas:
   $$X_{ind} \in \mathbb{R}^{(B \cdot M) 	imes L}$$
3. Estas series univariadas se procesan mediante un único codificador Transformer cuyos parámetros son compartidos por todos los canales. Esto reduce drásticamente el sobreajuste y la complejidad paramétrica.

### B. Segmentación por Parches (Patching)

En lugar de alimentar puntos individuales a la auto-atención (que es ruidosa y computacionalmente costosa), agrupamos timesteps adyacentes en parches locales. Para cada serie univariada de longitud $L$, extraemos parches de tamaño $P$ con un paso de salto $S$ (stride):
$$X_{patched} \in \mathbb{R}^{(B \cdot M) \times N \times P}$$
Donde el número de parches $N$ se define como:
$$N = \lfloor \frac{L - P}{S} \rfloor + 1$$

Para una secuencia de $L=128$, con un tamaño de parche $P=16$ y paso $S=8$, obtenemos $N=15$ parches de tamaño 16. 

**Ventaja principal:** La complejidad computacional del mecanismo de auto-atención pasa de ser cuadrática con respecto a los timesteps ($O(L^2)$) a ser cuadrática con respecto a los parches ($O(N^2)$). En este caso, de $128^2 = 16384$ operaciones a $15^2 = 225$ operaciones (aproximadamente **70 veces más rápido**).

### C. Proyección, Posición Aprendible y Regresión

1. **Proyección de Parche:** Cada parche se proyecta a la dimensión del Transformer $d_{model}$ mediante una capa lineal:
   $$Z_p = X_{patched} W_{proj} + b_{proj}$$
2. **Codificación Posicional Aprendible:** Se suma un tensor de parámetros entrenables para capturar el orden secuencial de los parches: $Z = Z_p + W_{pos}$, donde $W_{pos} \in \mathbb{R}^{1 \times N \times d_{model}}$.
3. **Transformer Encoder:** Se procesa $Z$ mediante las capas del codificador.
4. **Pooling y Mapeo Multicanal:**
   - Realizamos un promedio global sobre el eje de los parches $N$ para obtener un vector representativo por canal: $z_{pooled} \in \mathbb{R}^{(B \cdot M) \times d_{model}}$.
   - Re-separamos los canales y los aplanamos: $z_{flat} \in \mathbb{R}^{B \times (M \cdot d_{model})}$.
   - Una capa final de regresión proyecta $z_{flat}$ a los dos targets continuos de fatiga: $\mathbb{R}^{M \cdot d_{model}} \rightarrow \mathbb{R}^2$.

---

### Diagrama de Flujo de Tensores de PatchTST (Mermaid)

```mermaid
graph TD
    subgraph Entrada
        in["Input Fisiológico: (B, 128, M)"]
    end

    subgraph "Channel Independence"
        trans["Transpose(1, 2): (B, M, 128)"]
        reshape_ind["Reshape a lote univariado: (B*M, 128)"]
        in --> trans
        trans --> reshape_ind
    end

    subgraph "Manual Patching (unfold)"
        unfold["unfold(dim=-1, size=16, step=8)"]
        out_patched["Tensores Parcheados: (B*M, 15, 16)"]
        reshape_ind --> unfold
        unfold --> out_patched
    end

    subgraph "Proyección y Posición"
        proj["Linear Projection to d_model: (B*M, 15, d_model)"]
        pos["Añadir Positional Embedding (aprendible)"]
        out_patched --> proj
        proj --> pos
    end

    subgraph "Transformer Encoder (Pesos Compartidos)"
        enc["CustomTransformerEncoder (2 capas)"]
        out_enc["Latentes de parches: (B*M, 15, d_model)"]
        pos --> enc
        enc --> out_enc
    end

    subgraph "Cabezal de Salida (Flat & Regress)"
        pool["Mean Pooling sobre parches: (B*M, d_model)"]
        chan_sep["Reshape a canales separados: (B, M, d_model)"]
        flat["Aplanar Canales: (B, M * d_model)"]
        fc["Capa Lineal Regresora: (B, 2)"]
        
        out_enc --> pool
        pool --> chan_sep
        chan_sep --> flat
        flat --> fc
    end
```

---

### Citas Bibliográficas Científicas

* **Nie, Y., Nguyen, N. H., Sinthong, P., & Kalagnanam, J. (2022).** *A Time Series is Worth 64 Words: Long-term Forecasting with Transformers*. arXiv preprint arXiv:2211.14730. [Enlace al Paper](https://arxiv.org/abs/2211.14730)
* **Vaswani, A. et al. (2017).** *Attention Is All You Need*. Advances in Neural Information Processing Systems (NeurIPS), 30.

In [1]:
# SETUP e IMPORTACIONES
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
from fatigueset.models import CustomPatchTSTRegressor, FatigueSequenceDataset
from fatigueset.models.rnn import _prepare_target_table, _merge_raw_streams, _build_sequences

print("[OK] Imports completados y path configurado.")
print(f"Dispositivo actual: {'cuda' if torch.cuda.is_available() else 'cpu'}")

[OK] Imports completados y path configurado.
Dispositivo actual: cuda


## 2. Configuración del Pipeline y Construcción de Secuencias

Cargamos los datos fisiológicos del dataset `fatigueset` y alineamos los streams para generar ventanas temporales de 128 instantes de tiempo.

In [3]:
# Configuración del dataset y pipeline
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset...")
raw = pipeline.cargar_dataset(verbose=False)

print("Preparando targets del dataframe ML...")
df_ml = pipeline.construir_dataset_ml(raw)
df_targets = _prepare_target_table(df_ml)

print("Combinando streams fisiológicos crudos (Chest y Wrist)...")
df_raw = _merge_raw_streams(raw)

# Parámetros de ventanas de secuencia temporal
seq_len = 128
step = 32

print(f"Construyendo secuencias de tamaño={seq_len} y paso={step}...")
X_arr, y_arr, groups, feature_cols = _build_sequences(
    df_raw=df_raw,
    df_targets=df_targets,
    seq_len=seq_len,
    step=step
)

print(f"[OK] Dimensiones de tensores construidos:")
print(f"  - X: {X_arr.shape} (Número de ventanas x seq_len x features)")
print(f"  - y: {y_arr.shape} (Número de ventanas x 2 targets)")
print(f"  - Columnas de sensores: {len(feature_cols)}")

Cargando dataset...
Preparando targets del dataframe ML...
Combinando streams fisiológicos crudos (Chest y Wrist)...
Construyendo secuencias de tamaño=128 y paso=32...
[OK] Dimensiones de tensores construidos:
  - X: (1306, 128, 23) (Número de ventanas x seq_len x features)
  - y: (1306, 2) (Número de ventanas x 2 targets)
  - Columnas de sensores: 23


## 3. División de Datos por Participante (Group Split)

Dividimos los datos de manera que el participante `'01'` se reserve para validación y el resto para entrenamiento.

In [4]:
train_idx = np.where(groups != '01')[0]
val_idx = np.where(groups == '01')[0]

X_train, y_train = X_arr[train_idx], y_arr[train_idx]
X_val, y_val = X_arr[val_idx], y_arr[val_idx]

train_dataset = FatigueSequenceDataset(X_train, y_train)
val_dataset = FatigueSequenceDataset(X_val, y_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Train samples: 1184
Validation samples: 122


## 4. Inicialización del Regresor PatchTST

Instanciamos nuestro regresor PatchTST. Configuramos $d_{model} = 64$, 4 cabezas de atención, 2 bloques codificadores, un tamaño de parche de 16 y stride de 8.

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = len(feature_cols)
d_model = 64
num_heads = 4
num_layers = 2
dim_feedforward = 128
dropout = 0.1
patch_len = 16
stride = 8

model = CustomPatchTSTRegressor(
    input_size=input_size,
    patch_len=patch_len,
    stride=stride,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    output_size=2
).to(device)

print(model)

CustomPatchTSTRegressor(
  (linear_projection): Linear(in_features=16, out_features=64, bias=True)
  (dropout_layer): Dropout(p=0.1, inplace=False)
  (encoder_layers): ModuleList(
    (0-1): 2 x CustomTransformerEncoderLayer(
      (self_attn): CustomMultiHeadAttention(
        (q_linear): Linear(in_features=64, out_features=64, bias=True)
        (k_linear): Linear(in_features=64, out_features=64, bias=True)
        (v_linear): Linear(in_features=64, out_features=64, bias=True)
        (out_linear): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (linear1): Linear(in_features=64, out_features=128, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=128, out_features=64, bias=True)
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2):

## 5. Entrenamiento de Validación (5 Épocas)

Entrenamos el modelo durante 5 épocas empleando un optimizador Adam, una tasa de aprendizaje de $10^{-3}$ y gradient clipping para mantener la estabilidad del entrenamiento.

In [6]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("Iniciando entrenamiento...")

for epoch in range(1, epochs + 1):
    # Modo entrenamiento
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Modo validación
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    
    print(f"Epoch {epoch}/{epochs} - Train Loss (MSE): {avg_train_loss:.6f} - Val Loss (MSE): {avg_val_loss:.6f}")

print("[OK] Entrenamiento finalizado correctamente.")

Iniciando entrenamiento...
Epoch 1/5 - Train Loss (MSE): 613.486516 - Val Loss (MSE): 149.930176
Epoch 2/5 - Train Loss (MSE): 380.461860 - Val Loss (MSE): 179.613098
Epoch 3/5 - Train Loss (MSE): 363.206856 - Val Loss (MSE): 127.480361
Epoch 4/5 - Train Loss (MSE): 350.241163 - Val Loss (MSE): 156.302342
Epoch 5/5 - Train Loss (MSE): 337.268783 - Val Loss (MSE): 211.876333
[OK] Entrenamiento finalizado correctamente.


## 6. Serialización del Modelo

Guardamos los pesos del modelo en el directorio `/models/deep_learning/`.

In [7]:
output_dir = Path.cwd().parent / "models" / "deep_learning"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "patchtst_fatigue_notebook.pt"
torch.save(model.state_dict(), model_path)

print(f"[OK] Modelo guardado exitosamente en: {model_path}")

[OK] Modelo guardado exitosamente en: c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\deep_learning\patchtst_fatigue_notebook.pt
